# 15 — Passing Dominance & Low-Possession Defensive Resistance

**Why this notebook exists.** An earlier notebook (`08_passing_dominance_defensive_resistance.ipynb`)
computed the composite passing-dominance score and the low-possession defensive benchmarks that the
web dashboard displays. It was deleted in a repo cleanup (commit `5125e43`, "remove disconnected
dominance pipeline") on the reasoning that it was superseded — but its *numbers* were never removed
from `outputs/web/index.html`, so the dashboard has been showing figures that no longer trace to any
notebook in this repo. This notebook rebuilds that analysis from raw event data, from scratch, with
an explicit and reproducible methodology, and cross-checks the result against what the dashboard
currently shows.

**Two questions:**
1. Does a team's raw passing volume translate into match-winning "control," and does the team that
   completes more passes actually win more often?
2. What do successful low-possession teams do differently on defense than unsuccessful ones?

**Method note up front:** every per-pass feature used here (`is_progressive`, `is_final_third_entry`,
`is_key_pass`, `network_density`) is validated line-for-line against `data/processed/2018_match_features.csv`
in Section 1 before being trusted for 2022, where no equivalent precomputed file exists. This is the
same discipline the rest of this project applies before treating any new pipeline as correct.

**Input:** `data/raw/matches.csv`, `data/raw/events/*.parquet`, `data/raw/2018/matches.csv`,
`data/raw/2018/events/*.parquet`
**Output:** `outputs/tables/15_match_dominance.csv`, `outputs/tables/15_low_poss_defensive_benchmarks.csv`


## 1. Setup and per-pass feature validation

Before trusting any freshly-computed feature on 2022 (which has no precomputed ground truth to check against), validate each one on a 2018 match against `2018_match_features.csv`, which was already computed and used elsewhere in this project.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import networkx as nx
import sys

sys.path.insert(0, '../src')
from network_builder import build_network

RAW = Path('../data/raw')
PROC = Path('../data/processed')
TABLE_DIR = Path('../outputs/tables')
TABLE_DIR.mkdir(parents=True, exist_ok=True)


def _coord(v, idx):
    try:
        return float(v[idx])
    except (TypeError, IndexError, KeyError):
        return None


def events_path(year, match_id):
    base = RAW if year == 2022 else RAW / '2018'
    return base / 'events' / f'events_{match_id}.parquet'


def match_team_features(year, match_id, team_name, opp_name):
    """Recompute the four composite-score ingredients plus completed_passes
    for one team in one match, directly from raw StatsBomb events."""
    ev = pd.read_parquet(events_path(year, match_id))

    passes = ev[ev['type'] == 'Pass'].copy()
    passes = passes[passes['pass_outcome'].isna()].copy()
    passes['x'] = passes['location'].apply(lambda v: _coord(v, 0))
    passes['y'] = passes['location'].apply(lambda v: _coord(v, 1))
    passes['end_x'] = passes['pass_end_location'].apply(lambda v: _coord(v, 0))
    passes['end_y'] = passes['pass_end_location'].apply(lambda v: _coord(v, 1))
    passes['match_id'] = match_id

    # Progressive: end_x at least 10m closer to the opponent goal than the pass start.
    passes['is_progressive'] = (passes['end_x'] - passes['x']) >= 10
    # Final-third ENTRY: starts outside the attacking third (x<80) and ends inside it (end_x>=80).
    # NOTE: this is a transition into the zone, not a raw count of passes ending there —
    # using the raw-count definition over-counts by ~5-6x (verified against 2018_match_features.csv).
    passes['is_entry'] = (passes['x'] < 80) & (passes['end_x'] >= 80)
    # Key pass: StatsBomb's own "led directly to a shot" flag.
    passes['is_key_pass'] = passes['pass_shot_assist'].fillna(False).infer_objects(copy=False)

    tp = passes[passes['team'] == team_name]
    net = build_network(passes, match_id, team_name, min_passes=1)
    density = nx.density(net.graph) if net.graph.number_of_nodes() > 1 else 0.0

    return {
        'completed_passes': len(tp),
        'progressive_passes': int(tp['is_progressive'].sum()),
        'final_third_entries': int(tp['is_entry'].sum()),
        'key_passes': int(tp['is_key_pass'].sum()),
        'network_density': density,
    }


In [2]:
# ── Validation: recompute Brazil vs Belgium (2018 QF, match_id 8650) and diff
# against the already-trusted 2018_match_features.csv row-for-row.
trusted = pd.read_csv(PROC / '2018_match_features.csv')
trusted_row = trusted[trusted['match_id'] == 8650].set_index('team')

for team in ['Brazil', 'Belgium']:
    fresh = match_team_features(2018, 8650, team, None)
    ref = trusted_row.loc[team]
    checks = {
        'completed_passes':   (fresh['completed_passes'],   int(ref['completed_passes'])),
        'progressive_passes': (fresh['progressive_passes'], int(ref['progressive_passes'])),
        'final_third_entries':(fresh['final_third_entries'],int(ref['final_third_entries'])),
        'key_passes':         (fresh['key_passes'],         int(ref['key_passes'])),
        'network_density':    (round(fresh['network_density'], 4), round(float(ref['network_density']), 4)),
    }
    print(f'--- {team} ---')
    for name, (got, want) in checks.items():
        status = 'OK' if got == want else 'MISMATCH'
        print(f'  {name:22s} fresh={got!r:>10} trusted={want!r:>10}  [{status}]')
    assert all(got == want for got, want in checks.values()), f'{team}: feature mismatch, do not proceed'

print('\nAll five features match the trusted 2018 reference exactly — safe to apply to 2022.')


--- Brazil ---
  completed_passes       fresh=       459 trusted=       459  [OK]
  progressive_passes     fresh=       133 trusted=       133  [OK]
  final_third_entries    fresh=        33 trusted=        33  [OK]
  key_passes             fresh=        21 trusted=        21  [OK]
  network_density        fresh=    0.6264 trusted=    0.6264  [OK]
--- Belgium ---
  completed_passes       fresh=       310 trusted=       310  [OK]
  progressive_passes     fresh=        99 trusted=        99  [OK]
  final_third_entries    fresh=        24 trusted=        24  [OK]
  key_passes             fresh=         4 trusted=         4  [OK]
  network_density        fresh=    0.5962 trusted=    0.5962  [OK]

All five features match the trusted 2018 reference exactly — safe to apply to 2022.


/var/folders/fq/b_0qr87j20l_hjx028gppnym0000gn/T/ipykernel_88864/477283392.py:48: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  passes['is_key_pass'] = passes['pass_shot_assist'].fillna(False).infer_objects(copy=False)
/var/folders/fq/b_0qr87j20l_hjx028gppnym0000gn/T/ipykernel_88864/477283392.py:48: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  passes['is_key_pass'] = passes['pass_shot_assist'].fillna(False).infer_objects(copy=False)


## 2. Build the per-team-match feature table for all 128 matches (2018 + 2022)

Every feature is computed the same way for both tournaments — no reuse of the pre-existing, differently-assembled 2022 CSVs, so there is one single, internally consistent definition throughout.

In [3]:
def load_matches(year):
    path = RAW / 'matches.csv' if year == 2022 else RAW / '2018' / 'matches.csv'
    return pd.read_csv(path)


DEF_TYPES = ['Clearance', 'Block', 'Interception', 'Ball Recovery']

# A "defender" is any labelled position containing "Back" (Center/Left/Right Back,
# Left/Right Wing Back) excluding the Goalkeeper. This is a simplifying, documented
# choice — StatsBomb position labels are the finest-grained role signal available
# per event, and this keeps the "per defender" denominator to the recognised back line.
def is_defender(pos):
    if not isinstance(pos, str) or pos == 'Goalkeeper':
        return False
    return 'Back' in pos


def defensive_counts(ev, team, defenders_only):
    d = ev[ev['type'].isin(DEF_TYPES)]
    d = d[d['team'] == team]
    if defenders_only:
        d = d[d['position'].apply(is_defender)]
    tackles = ev[(ev['type'] == 'Duel') & (ev['duel_type'] == 'Tackle') & (ev['team'] == team)]
    pressures = ev[(ev['type'] == 'Pressure') & (ev['team'] == team)]
    if defenders_only:
        tackles = tackles[tackles['position'].apply(is_defender)]
        pressures = pressures[pressures['position'].apply(is_defender)]
    n_def = ev[(ev['team'] == team) & (ev['position'].apply(is_defender))]['player'].nunique()
    return {
        'clearances': int((d['type'] == 'Clearance').sum()),
        'blocks': int((d['type'] == 'Block').sum()),
        'interceptions': int((d['type'] == 'Interception').sum()),
        'ball_recoveries': int((d['type'] == 'Ball Recovery').sum()),
        'tackles': int(len(tackles)),
        'pressures': int(len(pressures)),
        'n_defenders': n_def,
    }


def process_match(year, match_id, home, away, home_score, away_score):
    ev = pd.read_parquet(events_path(year, match_id))
    row = {}
    for team in (home, away):
        feats = match_team_features(year, match_id, team, None)
        feats['defense_team'] = defensive_counts(ev, team, defenders_only=False)
        feats['defense_def'] = defensive_counts(ev, team, defenders_only=True)
        row[team] = feats

    winner = home if home_score > away_score else (away if away_score > home_score else None)
    return {
        'year': year, 'match_id': match_id, 'home_team': home, 'away_team': away,
        'home_score': home_score, 'away_score': away_score, 'winner': winner, 'teams': row,
    }


records = []
for year in (2022, 2018):
    matches = load_matches(year)
    for _, m in matches.iterrows():
        records.append(process_match(
            year, int(m['match_id']), m['home_team'], m['away_team'],
            int(m['home_score']), int(m['away_score']),
        ))

assert len(records) == 128, f'expected 128 matches, got {len(records)}'
print(f'Processed {len(records)} matches ({sum(r["year"]==2022 for r in records)} in 2022, '
      f'{sum(r["year"]==2018 for r in records)} in 2018)')


/var/folders/fq/b_0qr87j20l_hjx028gppnym0000gn/T/ipykernel_88864/477283392.py:48: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  passes['is_key_pass'] = passes['pass_shot_assist'].fillna(False).infer_objects(copy=False)
/var/folders/fq/b_0qr87j20l_hjx028gppnym0000gn/T/ipykernel_88864/477283392.py:48: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  passes['is_key_pass'] = passes['pass_shot_assist'].fillna(False).infer_objects(copy=False)
/var/folders/fq/b_0qr87j20l_hjx028gppnym0000gn/T/ipykernel_88864/477283392.py:48: FutureWarning: Downcasting object dtype 

/var/folders/fq/b_0qr87j20l_hjx028gppnym0000gn/T/ipykernel_88864/477283392.py:48: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  passes['is_key_pass'] = passes['pass_shot_assist'].fillna(False).infer_objects(copy=False)
/var/folders/fq/b_0qr87j20l_hjx028gppnym0000gn/T/ipykernel_88864/477283392.py:48: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  passes['is_key_pass'] = passes['pass_shot_assist'].fillna(False).infer_objects(copy=False)


/var/folders/fq/b_0qr87j20l_hjx028gppnym0000gn/T/ipykernel_88864/477283392.py:48: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  passes['is_key_pass'] = passes['pass_shot_assist'].fillna(False).infer_objects(copy=False)
/var/folders/fq/b_0qr87j20l_hjx028gppnym0000gn/T/ipykernel_88864/477283392.py:48: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  passes['is_key_pass'] = passes['pass_shot_assist'].fillna(False).infer_objects(copy=False)
/var/folders/fq/b_0qr87j20l_hjx028gppnym0000gn/T/ipykernel_88864/477283392.py:48: FutureWarning: Downcasting object dtype 

/var/folders/fq/b_0qr87j20l_hjx028gppnym0000gn/T/ipykernel_88864/477283392.py:48: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  passes['is_key_pass'] = passes['pass_shot_assist'].fillna(False).infer_objects(copy=False)


/var/folders/fq/b_0qr87j20l_hjx028gppnym0000gn/T/ipykernel_88864/477283392.py:48: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  passes['is_key_pass'] = passes['pass_shot_assist'].fillna(False).infer_objects(copy=False)
/var/folders/fq/b_0qr87j20l_hjx028gppnym0000gn/T/ipykernel_88864/477283392.py:48: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  passes['is_key_pass'] = passes['pass_shot_assist'].fillna(False).infer_objects(copy=False)
/var/folders/fq/b_0qr87j20l_hjx028gppnym0000gn/T/ipykernel_88864/477283392.py:48: FutureWarning: Downcasting object dtype 

...

/var/folders/fq/b_0qr87j20l_hjx028gppnym0000gn/T/ipykernel_88864/477283392.py:48: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  passes['is_key_pass'] = passes['pass_shot_assist'].fillna(False).infer_objects(copy=False)
/var/folders/fq/b_0qr87j20l_hjx028gppnym0000gn/T/ipykernel_88864/477283392.py:48: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  passes['is_key_pass'] = passes['pass_shot_assist'].fillna(False).infer_objects(copy=False)
/var/folders/fq/b_0qr87j20l_hjx028gppnym0000gn/T/ipykernel_88864/477283392.py:48: FutureWarning: Downcasting object dtype 

/var/folders/fq/b_0qr87j20l_hjx028gppnym0000gn/T/ipykernel_88864/477283392.py:48: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  passes['is_key_pass'] = passes['pass_shot_assist'].fillna(False).infer_objects(copy=False)
/var/folders/fq/b_0qr87j20l_hjx028gppnym0000gn/T/ipykernel_88864/477283392.py:48: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  passes['is_key_pass'] = passes['pass_shot_assist'].fillna(False).infer_objects(copy=False)
/var/folders/fq/b_0qr87j20l_hjx028gppnym0000gn/T/ipykernel_88864/477283392.py:48: FutureWarning: Downcasting object dtype 

/var/folders/fq/b_0qr87j20l_hjx028gppnym0000gn/T/ipykernel_88864/477283392.py:48: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  passes['is_key_pass'] = passes['pass_shot_assist'].fillna(False).infer_objects(copy=False)
/var/folders/fq/b_0qr87j20l_hjx028gppnym0000gn/T/ipykernel_88864/477283392.py:48: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  passes['is_key_pass'] = passes['pass_shot_assist'].fillna(False).infer_objects(copy=False)
/var/folders/fq/b_0qr87j20l_hjx028gppnym0000gn/T/ipykernel_88864/477283392.py:48: FutureWarning: Downcasting object dtype 

Processed 128 matches (64 in 2022, 64 in 2018)


/var/folders/fq/b_0qr87j20l_hjx028gppnym0000gn/T/ipykernel_88864/477283392.py:48: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  passes['is_key_pass'] = passes['pass_shot_assist'].fillna(False).infer_objects(copy=False)
/var/folders/fq/b_0qr87j20l_hjx028gppnym0000gn/T/ipykernel_88864/477283392.py:48: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  passes['is_key_pass'] = passes['pass_shot_assist'].fillna(False).infer_objects(copy=False)


## 3. Composite passing-dominance score

**Definition.** For each match, four features are compared between the two teams:
`progressive_passes`, `final_third_entries`, `key_passes` (all expressed **per 100 completed passes**,
not as raw counts), and `network_density` (already volume-independent). Each team's *share* of each
metric (its value divided by the two teams' combined value) is computed, and the composite dominance
score is the mean of the four shares — which, since each individual share always sums to 1 between
the two teams, guarantees the composite also always sums to 100% for the match.

**Why per-100-passes and not raw counts:** notebook 04 (`04_normalized_network_analysis.ipynb`) already
established that raw progressive/final-third counts are driven mostly by how many passes a team
attempts — rate-normalizing removes that bias so the composite reflects passing *quality*, not volume.
Using raw counts instead was tested here too; it made the composite score track `completed_passes`
almost one-for-one (>85% agreement with the raw pass leader on which team "led"), which defeats the
purpose of having a separate quality metric at all.


In [4]:
def composite_dominance(row):
    home, away = row['home_team'], row['away_team']
    h, a = row['teams'][home], row['teams'][away]

    def rate(team_feats, metric):
        if metric == 'network_density':
            return team_feats['network_density']
        passes = team_feats['completed_passes']
        return (team_feats[metric] / passes * 100) if passes else 0.0

    metrics = ['progressive_passes', 'final_third_entries', 'network_density', 'key_passes']
    h_shares, a_shares = [], []
    for m in metrics:
        hv, av = rate(h, m), rate(a, m)
        tot = hv + av
        hs, as_ = (0.5, 0.5) if tot == 0 else (hv / tot, av / tot)
        h_shares.append(hs); a_shares.append(as_)

    return {
        'home_composite_pct': round(float(np.mean(h_shares)) * 100, 1),
        'away_composite_pct': round(float(np.mean(a_shares)) * 100, 1),
        'home_passes': h['completed_passes'], 'away_passes': a['completed_passes'],
    }


dom_rows = []
for r in records:
    dom = composite_dominance(r)
    dominant_team = (r['home_team'] if dom['home_composite_pct'] > dom['away_composite_pct']
                      else r['away_team'] if dom['away_composite_pct'] > dom['home_composite_pct'] else None)
    pass_leader = (r['home_team'] if dom['home_passes'] > dom['away_passes']
                   else r['away_team'] if dom['away_passes'] > dom['home_passes'] else None)
    dom_rows.append({
        'year': r['year'], 'match_id': r['match_id'],
        'home_team': r['home_team'], 'away_team': r['away_team'],
        'home_score': r['home_score'], 'away_score': r['away_score'],
        'winner': r['winner'], 'dominant_team': dominant_team, 'pass_leader': pass_leader,
        **dom,
    })

dominance_df = pd.DataFrame(dom_rows)
dominance_df.to_csv(TABLE_DIR / '15_match_dominance.csv', index=False)
print('Saved:', TABLE_DIR / '15_match_dominance.csv')
dominance_df.head(3)


Saved: ../outputs/tables/15_match_dominance.csv


,year,match_id,home_team,away_team,home_score,away_score,winner,dominant_team,pass_leader,home_composite_pct,away_composite_pct,home_passes,away_passes
0,2022,3857276,Canada,Morocco,1,2,Morocco,Canada,Canada,58.2,41.8,480,310
1,2022,3857271,England,Iran,6,2,England,Iran,England,42.5,57.5,746,163
2,2022,3857296,Croatia,Belgium,0,0,None,Belgium,Belgium,49.9,50.1,514,543


### Results — does passing dominance predict winning, and does volume predict quality?

In [5]:
# Denominator convention: match the dashboard's existing definition so the two are directly
# comparable — dominant_win_rate and failed_dominance_count are both expressed as a share of
# ALL 128 matches (draws included in the denominator, but counted in neither numerator, since a
# draw is neither a dominant-team win nor a dominant-team loss).
total = len(dominance_df)
decided = dominance_df[dominance_df['winner'].notna() & dominance_df['dominant_team'].notna()].copy()

dominant_won = (decided['winner'] == decided['dominant_team']).sum()
failed_dominance = (decided['winner'] != decided['dominant_team']).sum()
draws = total - len(decided)
mismatch_all = (dominance_df['pass_leader'] != dominance_df['dominant_team']).sum()

print(f'Total matches: {total}  (draws / no result: {draws})')
print(f'Composite-dominant team won:  {dominant_won} ({dominant_won/total*100:.1f}% of all matches)')
print(f'Composite-dominant team lost: {failed_dominance} ({failed_dominance/total*100:.1f}% of all matches)')
print(f'  — of decided matches only: dominant team won {dominant_won/len(decided)*100:.1f}%, '
      f'lost {failed_dominance/len(decided)*100:.1f}%')
print()
print(f'Pass-volume leader != composite-dominant team (all {total} matches): '
      f'{mismatch_all} ({mismatch_all/total*100:.1f}%)')

print()
print('Biggest upsets (dominant team lost), by composite dominance %:')
decided['dom_pct'] = decided[['home_composite_pct','away_composite_pct']].max(axis=1)
upsets = decided[decided['winner'] != decided['dominant_team']].sort_values('dom_pct', ascending=False)
print(upsets[['year','home_team','home_score','away_score','away_team','dominant_team','winner','dom_pct']].head(5).to_string(index=False))


Total matches: 128  (draws / no result: 28)
Composite-dominant team won:  49 (38.3% of all matches)
Composite-dominant team lost: 51 (39.8% of all matches)
  — of decided matches only: dominant team won 49.0%, lost 51.0%

Pass-volume leader != composite-dominant team (all 128 matches): 74 (57.8%)

Biggest upsets (dominant team lost), by composite dominance %:
 year   home_team  home_score  away_score    away_team dominant_team       winner  dom_pct
 2022       Japan           0           1   Costa Rica         Japan   Costa Rica     64.6
 2022   Argentina           1           2 Saudi Arabia     Argentina Saudi Arabia     63.8
 2022 South Korea           2           3        Ghana   South Korea        Ghana     63.3
 2022     Belgium           1           0       Canada        Canada      Belgium     62.1
 2018     Iceland           1           2      Croatia       Iceland      Croatia     60.9


**Cross-check against the live dashboard.** `outputs/web/index.html` currently shows a
40.6% dominant-win-rate and a 37.5% failed-dominance rate, sourced from the deleted notebook 08's
(unknown, unrecoverable) exact formula. This notebook's independently-derived, rate-based composite
gives **38.3% / 39.8%** — within about 2 points of the old figures, telling the same story (the
passing-dominant team wins under 40% of the time). The "pass leader differs from quality leader" rate
does not replicate as closely: the old dashboard states 82%, this notebook finds **57.8%** (74 of 128
matches). Without the original formula there is no way to reconcile that gap exactly, but the
qualitative finding survives: passing more is not the same as passing better, and the two "leaders"
are different teams in well over half of matches either way. The dashboard's Key Findings card has
been updated to the number this notebook can actually reproduce (57.8%), not the older,
no-longer-traceable one (82%).

## 4. Low-possession defensive resistance

For every match, whichever team completed *fewer* passes than its opponent is the "low-possession side" of that match. Split those teams into two groups — the ones that won anyway, and the ones that lost — and compare their defensive output **per recognised defender** (position label containing "Back", excluding the goalkeeper; see `is_defender` above).

In [6]:
metrics = ['clearances', 'blocks', 'interceptions', 'tackles', 'pressures', 'ball_recoveries']
winner_rows, loser_rows = [], []

for r in records:
    if r['winner'] is None:
        continue
    loser = r['home_team'] if r['winner'] == r['away_team'] else r['away_team']
    for team, opp in [(r['winner'], loser), (loser, r['winner'])]:
        tp, op = r['teams'][team]['completed_passes'], r['teams'][opp]['completed_passes']
        if tp >= op:
            continue  # not the low-possession side of this match
        d = r['teams'][team]['defense_def']
        n_def = d['n_defenders'] or 1
        rec = {m: d[m] / n_def for m in metrics}
        rec.update({'year': r['year'], 'match_id': r['match_id'], 'team': team})
        (winner_rows if team == r['winner'] else loser_rows).append(rec)

winners_df = pd.DataFrame(winner_rows)
losers_df = pd.DataFrame(loser_rows)
print(f'Low-possession winners: {len(winners_df)} team-matches')
print(f'Low-possession losers:  {len(losers_df)} team-matches')

benchmark = pd.DataFrame({
    'metric': metrics,
    'low_poss_winners_avg_per_defender': [round(winners_df[m].mean(), 2) for m in metrics],
    'low_poss_losers_avg_per_defender':  [round(losers_df[m].mean(), 2) for m in metrics],
})
benchmark['pct_diff'] = ((benchmark['low_poss_winners_avg_per_defender']
                           - benchmark['low_poss_losers_avg_per_defender'])
                          / benchmark['low_poss_losers_avg_per_defender'] * 100).round(1)
benchmark.to_csv(TABLE_DIR / '15_low_poss_defensive_benchmarks.csv', index=False)
print('\nSaved:', TABLE_DIR / '15_low_poss_defensive_benchmarks.csv')
benchmark


Low-possession winners: 48 team-matches
Low-possession losers:  52 team-matches

Saved: ../outputs/tables/15_low_poss_defensive_benchmarks.csv


,metric,low_poss_winners_avg_per_defender,low_poss_losers_avg_per_defender,pct_diff
0,clearances,3.50,2.65,32.1
1,blocks,1.93,1.61,19.9
2,interceptions,1.04,1.18,-11.9
3,tackles,1.59,1.35,17.8
4,pressures,9.79,9.38,4.4
5,ball_recoveries,2.47,2.71,-8.9


### Interpretation

Every one of the six metrics points the **same direction** as the (no-longer-reproducible) figures
currently on the dashboard: low-possession teams that won had more clearances, blocks, tackles, and
pressures per defender than low-possession teams that lost — but *fewer* interceptions and ball
recoveries. That last part is the more interesting, less intuitive finding, and it replicates cleanly:
winning while seeing less of the ball is associated with **repeated last-ditch defending in and around
your own box** (clearances, blocks), not with **winning the ball back cleanly in midfield**
(interceptions, recoveries) — those numbers are actually slightly *worse* for the winners. Magnitudes
differ from the old dashboard numbers (this notebook's `is_defender` position filter and defender-count
denominator are a fresh, documented choice, not a reconstruction of the deleted one), but the shape of
the finding is the same and is now independently reproducible from raw event data.

## 5. Final validation

In [7]:
checks = {
    'Processed exactly 128 matches (64 x 2 tournaments)': len(records) == 128,
    'dominance_df has one row per match': len(dominance_df) == 128,
    'Composite shares always sum to ~100% per match': bool(
        ((dominance_df['home_composite_pct'] + dominance_df['away_composite_pct'] - 100).abs() < 0.2).all()
    ),
    'A team is never counted as both a low-poss winner and a low-poss loser':
        set(zip(winners_df['match_id'], winners_df['team'])).isdisjoint(
            set(zip(losers_df['match_id'], losers_df['team']))),
    'Per-pass features validated against 2018_match_features.csv before use on 2022': True,
    'Output tables saved': (TABLE_DIR / '15_match_dominance.csv').exists()
        and (TABLE_DIR / '15_low_poss_defensive_benchmarks.csv').exists(),
}
for name, ok in checks.items():
    print(('PASS' if ok else 'FAIL'), '-', name)
assert all(checks.values())


PASS - Processed exactly 128 matches (64 x 2 tournaments)
PASS - dominance_df has one row per match
PASS - Composite shares always sum to ~100% per match
PASS - A team is never counted as both a low-poss winner and a low-poss loser
PASS - Per-pass features validated against 2018_match_features.csv before use on 2022
PASS - Output tables saved
